<a href="https://colab.research.google.com/github/binghan1123/1142-programming-language/blob/main/HW4_PTT_GoogleSheet_RAG%E6%95%B4%E7%90%86%E7%89%88.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HW4：PTT → Google Sheet → RAG（整理版）

這份 notebook 保留完整流程：

1. 爬取 PTT movie 文章
2. 寫入指定 Google Sheet
3. 從 Google Sheet 讀回資料
4. 建立 FAISS RAG 索引
5. 用 Gemini 根據 PTT 資料回答問題

主要修正：原本設定了 `SHEET_URL`，但實際用 `gc.open(WORKSHEET_NAME)` 開啟試算表，容易打開錯的 Spreadsheet。新版固定使用 `gc.open_by_url(SHEET_URL)`。


In [1]:
# 安裝必要套件
!pip -q install gspread gspread-dataframe faiss-cpu sentence-transformers beautifulsoup4 requests google-generativeai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 49.3 MB/s eta 0:00:00


In [2]:
import re
import time
import uuid
from datetime import datetime
from urllib.parse import urljoin

import requests
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup

import faiss
from sentence_transformers import SentenceTransformer

import google.generativeai as genai
from google.colab import auth, userdata
import gspread
from google.auth import default
from gspread_dataframe import set_with_dataframe, get_as_dataframe


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


## 1. 基本設定

請確認 `SHEET_URL` 是你要寫入的 Google Sheet。  
`PTT_WORKSHEET_NAME` 是存放 PTT 原始文章的分頁。


In [3]:
SHEET_URL = "https://docs.google.com/spreadsheets/d/1yefD7L8LSK27T-B_V4PWyk6tEUl5IGUCs9Lw8SbCGF0/edit?usp=sharing"
PTT_WORKSHEET_NAME = "工作表4"
TIMEZONE_NOTE = "Asia/Taipei"

PTT_HEADER = [
    "post_id", "title", "url", "date", "author", "nrec",
    "created_at", "fetched_at", "content"
]

PTT_STOCK_INDEX = "https://www.ptt.cc/bbs/Stock/index.html"
PTT_COOKIES = {"over18": "1"}
USER_AGENT = "Mozilla/5.0 (compatible; Colab PTT crawler)"


## 2. 連線 Google Sheet

這裡是最重要的修正：使用 `open_by_url(SHEET_URL)`，不要用 worksheet 名稱打開 spreadsheet。


In [4]:
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# 關鍵修正：直接用網址開啟指定 Google Sheet
sh = gc.open_by_url(SHEET_URL)
print(f"✅ 已開啟試算表：{sh.title}")
print(f"🔗 {SHEET_URL}")


✅ 已開啟試算表：程式語言測試資料
🔗 https://docs.google.com/spreadsheets/d/1yefD7L8LSK27T-B_V4PWyk6tEUl5IGUCs9Lw8SbCGF0/edit?usp=sharing


In [5]:
def ensure_worksheet(spreadsheet, title, header, rows=1000):
    """取得或建立 worksheet，並確保表頭正確。"""
    try:
        ws = spreadsheet.worksheet(title)
    except gspread.WorksheetNotFound:
        ws = spreadsheet.add_worksheet(title=title, rows=str(rows), cols=str(len(header) + 5))
        ws.update([header])
        return ws

    values = ws.get_all_values()
    if not values:
        ws.update([header])
    elif values[0] != header:
        # 保留資料但重建欄位較危險，因此這裡直接清掉並重新建立正確表頭。
        # 若你要保留舊資料，請先備份 Google Sheet。
        ws.clear()
        ws.update([header])
    return ws


def read_sheet_df(ws, header):
    """從 worksheet 讀成 DataFrame，並清掉空列。"""
    df = get_as_dataframe(ws, evaluate_formulas=True, dtype=str).dropna(how="all")
    if df.empty:
        return pd.DataFrame(columns=header)
    df = df.loc[:, [c for c in df.columns if not str(c).startswith("Unnamed")]]
    for col in header:
        if col not in df.columns:
            df[col] = ""
    return df[header].fillna("")


def write_sheet_df(ws, df, header):
    """把 DataFrame 寫回 worksheet。"""
    df_out = df.copy()
    for col in header:
        if col not in df_out.columns:
            df_out[col] = ""
    df_out = df_out[header].infer_objects(copy=False).fillna("") # Add infer_objects to address FutureWarning

    # Google Sheet 寫入前統一轉字串，避免 Timestamp / NaN 型別問題
    for c in df_out.columns:
        df_out[c] = df_out[c].astype(str)

    ws.clear()
    set_with_dataframe(ws, df_out, include_index=False, include_column_header=True, resize=True)
    return len(df_out)


ws_ptt = ensure_worksheet(sh, PTT_WORKSHEET_NAME, PTT_HEADER)
print(f"✅ 已準備 worksheet：{ws_ptt.title}")

✅ 已準備 worksheet：工作表4


## 3. PTT movie 爬蟲

這段只負責爬 PTT，不碰 RAG。資料會先存在 `new_posts_df`。


In [6]:
def now_iso():
    return datetime.now().isoformat(timespec="seconds")


def get_soup(url):
    resp = requests.get(
        url,
        timeout=20,
        headers={"User-Agent": USER_AGENT},
        cookies=PTT_COOKIES,
    )
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")


def get_prev_index_url(soup):
    for a in soup.select("div.btn-group-paging a.btn.wide"):
        if "上頁" in a.get_text(strip=True):
            href = a.get("href")
            return urljoin("https://www.ptt.cc", href) if href else None
    return None


def parse_nrec(nrec_span):
    if not nrec_span:
        return 0
    txt = nrec_span.get_text(strip=True)
    if txt == "爆":
        return 100
    if txt.startswith("X"):
        try:
            return -int(txt[1:])
        except Exception:
            return -10
    try:
        return int(txt)
    except Exception:
        return 0


def extract_post_list(index_soup):
    posts = []
    for item in index_soup.select("div.r-ent"):
        a = item.select_one("div.title a")
        if not a:
            continue

        title = a.get_text(strip=True)
        url = urljoin("https://www.ptt.cc", a.get("href"))
        author_node = item.select_one("div.author")
        date_node = item.select_one("div.date")
        nrec_node = item.select_one("div.nrec span")

        posts.append({
            "title": title,
            "url": url,
            "author": author_node.get_text(strip=True) if author_node else "",
            "date": date_node.get_text(strip=True) if date_node else "",
            "nrec": parse_nrec(nrec_node),
        })
    return posts


def clean_ptt_content(article_soup):
    main = article_soup.select_one("#main-content")
    if not main:
        return "", ""

    # 取出文章建立時間
    created_at = ""
    metalines = main.select("div.article-metaline")
    for m in metalines:
        tag = m.select_one("span.article-meta-tag")
        value = m.select_one("span.article-meta-value")
        if tag and value and tag.get_text(strip=True) == "時間":
            created_at = value.get_text(strip=True)

    # 移除 meta 與推文
    for node in main.select("div.article-metaline, div.article-metaline-right, div.push"):
        node.decompose()

    text = main.get_text("\n", strip=True)
    text = re.split(r"\n--\n|\n※ 發信站:", text)[0].strip()
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text, created_at


def make_post_id(url):
    # 用文章網址檔名當 post_id，穩定且方便去重
    return url.rstrip("/").split("/")[-1].replace(".html", "")


def crawl_ptt_movie(pages=5, delay=0.5):
    """爬取 PTT movie 最新 pages 頁文章。"""
    all_rows = []
    index_url = PTT_STOCK_INDEX

    for page in range(int(pages)):
        print(f"📄 正在讀取列表頁 {page + 1}/{pages}: {index_url}")
        index_soup = get_soup(index_url)
        post_list = extract_post_list(index_soup)

        for p in post_list:
            try:
                article_soup = get_soup(p["url"])
                content, created_at = clean_ptt_content(article_soup)
                row = {
                    "post_id": make_post_id(p["url"]),
                    "title": p["title"],
                    "url": p["url"],
                    "date": p["date"],
                    "author": p["author"],
                    "nrec": p["nrec"],
                    "created_at": created_at,
                    "fetched_at": now_iso(),
                    "content": content,
                }
                all_rows.append(row)
                time.sleep(delay)
            except Exception as e:
                print(f"⚠️ 跳過文章：{p.get('title', '')}，原因：{e}")

        prev_url = get_prev_index_url(index_soup)
        if not prev_url:
            break
        index_url = prev_url
        time.sleep(delay)

    df = pd.DataFrame(all_rows, columns=PTT_HEADER)
    print(f"✅ 本次爬到 {len(df)} 篇文章")
    return df

## 4. 執行爬蟲並寫入 Google Sheet

這一格會：

1. 從 Google Sheet 讀取既有資料
2. 爬取新的 PTT 資料
3. 合併並用 `post_id` 去重
4. 寫回 Google Sheet
5. 再讀一次確認真的寫入成功


In [8]:
try:
    # 修改此處的 pages=5，以對應您在函式定義中的期望頁數
    new_posts_df = crawl_ptt_movie(pages=5, delay=2.0)
except Exception as e:
    print(f"❌ 爬蟲過程發生連線錯誤：{e}")
    print("💡 建議：請稍候再試，或增加 crawl_ptt_movie 中的 delay 參數。")
    new_posts_df = pd.DataFrame(columns=PTT_HEADER)

old_posts_df = read_sheet_df(ws_ptt, PTT_HEADER)
print(f"📌 Google Sheet 原本有 {len(old_posts_df)} 筆")

if not new_posts_df.empty:
    ptt_posts_df = pd.concat([old_posts_df, new_posts_df], ignore_index=True)
    ptt_posts_df = ptt_posts_df.drop_duplicates(subset=["post_id"], keep="last")
    ptt_posts_df = ptt_posts_df.sort_values(by="fetched_at", ascending=False)

    written_count = write_sheet_df(ws_ptt, ptt_posts_df, PTT_HEADER)
    print(f"✅ 已寫入 Google Sheet：{written_count} 筆")

    verify_df = read_sheet_df(ws_ptt, PTT_HEADER)
    print(f"🔍 從 Google Sheet 重新讀回：{len(verify_df)} 筆")

    if len(verify_df) == written_count:
        print("✅ 寫入驗證成功")
    else:
        print("⚠️ 寫入筆數與讀回筆數不同，請檢查 Google Sheet 權限或資料格式")
else:
    print("ℹ️ 本次沒有新抓取到資料，跳過寫入步驟。")

📄 正在讀取列表頁 1/5: https://www.ptt.cc/bbs/Stock/index.html
📄 正在讀取列表頁 2/5: https://www.ptt.cc/bbs/Stock/index10089.html
📄 正在讀取列表頁 3/5: https://www.ptt.cc/bbs/Stock/index10088.html
📄 正在讀取列表頁 4/5: https://www.ptt.cc/bbs/Stock/index10087.html
📄 正在讀取列表頁 5/5: https://www.ptt.cc/bbs/Stock/index10086.html
✅ 本次爬到 96 篇文章
📌 Google Sheet 原本有 166 筆
✅ 已寫入 Google Sheet：177 筆
🔍 從 Google Sheet 重新讀回：177 筆
✅ 寫入驗證成功


## 5. 從 Google Sheet 建立 RAG 索引

重點：RAG 不直接吃剛爬下來的記憶體資料，而是**從 Google Sheet 重新讀回**，這樣才能確認流程真的是：

`PTT → Google Sheet → RAG`


In [9]:
# 從 Google Sheet 重新讀取，作為 RAG 的唯一資料來源
rag_source_df = read_sheet_df(ws_ptt, PTT_HEADER)

# 將 nrec 欄位轉換為數字，並將無法轉換的值設為 NaN，然後填充為 0
rag_source_df['nrec'] = pd.to_numeric(rag_source_df['nrec'], errors='coerce').fillna(0).astype(int)

# 只保留推文數大於的文章
rag_source_df = rag_source_df[rag_source_df["nrec"] >= 5].copy()
print(f"📚 可用於 RAG 的文章數：{len(rag_source_df)}")

rag_source_df.head()

📚 可用於 RAG 的文章數：153


,post_id,title,url,date,author,nrec,created_at,fetched_at,content
0,M.1780615247.A.DFC,[請益] 期貨止損的紀律 大概抓多少好,https://www.ptt.cc/bbs/Stock/M.1780615247.A.DF...,6/5,DivaNight,100,Fri Jun 5 07:20:43 2026,2026-06-05 11:02:47,----------------------------------------------...
1,M.1780593263.A.218,Re: [新聞] 魏哲家私下抱怨「AI爆發怎不早講」！ 黃,https://www.ptt.cc/bbs/Stock/M.1780593263.A.21...,6/5,Subaru5566,6,Fri Jun 5 01:14:21 2026,2026-06-05 11:02:44,https://www.youtube.com/watch?v=lMz9_YNg1Wk\n是...
2,M.1780590418.A.A48,[心得] 正二vs主動式vs安聯,https://www.ptt.cc/bbs/Stock/M.1780590418.A.A4...,6/5,yamaha449,29,Fri Jun 5 00:26:56 2026,2026-06-05 11:02:42,上一篇提到\n正二(685) vs 981A\n但是過去沒有981A熊市的樣本\n就拿安聯來...
3,M.1780587788.A.521,[新聞] 中共首次將個人直接對境外投資納入監控,https://www.ptt.cc/bbs/Stock/M.1780587788.A.52...,6/4,Su22,100,Thu Jun 4 23:43:05 2026,2026-06-05 11:02:40,原文標題：\n中共首次將個人直接對境外投資納入監控 引關注\n原文連結：\nhttps://...
4,M.1780586373.A.5FE,Re: [請益] 台灣不收證所稅而收證交稅是劫貧濟富？,https://www.ptt.cc/bbs/Stock/M.1780586373.A.5F...,6/4,Historia,29,Thu Jun 4 23:19:08 2026,2026-06-05 11:02:38,※ 引述《sonans ()》之銘言：\n: 讀周冠男教授的書長期買進時，\n: 他批評台灣...


In [10]:
print("正在載入多語言 Embedding 模型...")
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("✅ Embedding 模型載入完成")


def build_rag_documents(df):
    docs = []
    for _, row in df.iterrows():
        title = str(row.get("title", ""))
        content = str(row.get("content", ""))
        url = str(row.get("url", ""))
        author = str(row.get("author", ""))
        date = str(row.get("date", ""))
        nrec = str(row.get("nrec", ""))

        text = (f"標題：{title}\n"
                f"作者：{author}\n"
                f"日期：{date}\n"
                f"推文數：{nrec}\n"
                f"內容：{content}")
        docs.append({
            "post_id": str(row.get("post_id", "")),
            "title": title,
            "url": url,
            "text": text,
        })
    return docs


def build_faiss_index(docs):
    if not docs:
        raise ValueError("沒有可建立索引的文件")

    texts = [d["text"] for d in docs]
    embeddings = embedding_model.encode(texts, convert_to_numpy=True, show_progress_bar=True)
    embeddings = embeddings.astype("float32")

    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)
    return index, embeddings


rag_documents = build_rag_documents(rag_source_df)
rag_index, rag_embeddings = build_faiss_index(rag_documents)

print(f"✅ RAG 索引建立完成：{len(rag_documents)} 篇文章，向量維度 {rag_embeddings.shape[1]}")

正在載入多語言 Embedding 模型...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding 模型載入完成


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

✅ RAG 索引建立完成：153 篇文章，向量維度 384


## 6. Gemini 設定與 RAG 問答

請先在 Colab Secrets 裡建立 `gemini`，內容是你的 Gemini API key。


In [11]:
api_key = userdata.get("41371209H")
if not api_key:
    raise ValueError("找不到 Colab Secret：gemini。請先在 Colab Secrets 新增 Gemini API key。")

genai.configure(api_key=api_key)

# 若你的帳號不支援這個模型，可改成你可用的 Gemini model name
GEMINI_MODEL_NAME = "gemini-2.5-flash-lite"
llm = genai.GenerativeModel(GEMINI_MODEL_NAME)
print(f"✅ Gemini 已設定：{GEMINI_MODEL_NAME}")


✅ Gemini 已設定：gemini-2.5-flash-lite


In [12]:
def retrieve_docs(query, k=5):
    if rag_index is None or not rag_documents:
        return []
    q_emb = embedding_model.encode([query], convert_to_numpy=True).astype("float32")
    distances, indices = rag_index.search(q_emb, int(k))

    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if idx == -1:
            continue
        doc = rag_documents[idx].copy()
        doc["distance"] = float(dist)
        results.append(doc)
    return results


def query_rag(question, k=10):
    docs = retrieve_docs(question, k=k)
    if not docs:
        return "找不到相關 PTT 資料。"

    context = "\n\n---\n\n".join(
        [f"來源標題：{d['title']}\n來源網址：{d['url']}\n{d['text']}" for d in docs]
    )

    prompt = f"""

你是一個專業的投資分析師，你的任務是回答關於【PTT 股市版 (Stock)】的投資相關問題。
請根據以下 PTT 資料，幫我條列式總結重點。
如果是請益文，請幫我整理出多空雙方的論點。
個股健檢幫我分析這隻股票的基本面、籌碼面等等
...

### 嚴格規範：
1. 你「僅限」使用提供的【PTT 資料】內容回答。

2. 判斷邏輯：如果問題是關於「電影推薦」、「天氣」、「美食」或任何非關「投資/股票/市場趨勢」的內容，即便資料中包含該關鍵字，你也必須直接回覆：「抱歉，我只能回答與PPT股市版內容相關的問題」。
3. 若資料中完全未提及問題的具體投資答案，也請回覆：「抱歉，我只能回答與PPT股市版內容相關的問題」。
4. 若資料中未提及問題答案，或問題與 PTT 股市資料無關（例如問天氣、問基礎定義、問其他品牌），請**嚴格遵守**僅回覆：「抱歉，我只能回答與PPT股市版內容相關的問題」。
5. 禁止發揮想像力或使用訓練數據中的外部知識。
【PTT 資料】
{context}

【問題】
{question}

【回答】
""".strip()

    response = llm.generate_content(prompt)
    return response.text

## 7. 快速測試


In [15]:
question = input("請輸入問題：")
answer = query_rag(question, k=15)
print(answer)


請輸入問題：新上映的電影
抱歉，我只能回答與PPT股市版內容相關的問題。


In [16]:
import os
import re
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from collections import Counter, defaultdict
import google.generativeai as genai

# ==========================================
# 0. 基礎設定與 API Key 配置
# ==========================================
# 請確保你的 Colab 鑰匙圈（Secrets）裡有設定 'GEMINI_API_KEY'
from google.colab import userdata
try:
    GOOGLE_API_KEY = userdata.get('41371209H')
    genai.configure(api_key=GOOGLE_API_KEY)
    # 使用適合處理複雜邏輯與長文本的 Gemini 2.5 Pro
    model = genai.GenerativeModel('gemini-2.5-flash-lite')
except Exception as e:
    print(f"❌ API Key 設定失敗，請檢查 Colab 左側鑰匙圈：{e}")

# ==========================================
# 1. PTT 股版爬蟲模組
# ==========================================
def crawl_ptt_stock(pages=3, min_push=10):
    print(f"🕷️ 開始爬取 PTT 股版最新 {pages} 頁的文章 (過濾低於 {min_push} 推的文章)...")
    url = "https://www.ptt.cc/bbs/Stock/index.html"
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
    cookies = {'over18': '1'}

    posts_data = []

    for _ in range(pages):
        res = requests.get(url, headers=headers, cookies=cookies)
        if res.status_code != 200:
            print(f"⚠️ 無法讀取網頁: {url}")
            break

        soup = BeautifulSoup(res.text, 'html.parser')

        # 尋找文章列表
        divs = soup.find_all('div', class_='r-ent')
        for div in divs:
            try:
                # 篩選推文數
                push_str = div.find('div', class_='nrec').text.strip()
                push_count = 0
                if push_str:
                    if push_str == '爆': push_count = 100
                    elif push_str.startswith('X'): push_count = -10 # 噓文先忽略
                    else: push_count = int(push_str)

                if push_count < min_push:
                    continue

                title_a = div.find('div', class_='title').find('a')
                if not title_a: continue

                title = title_a.text.strip()
                # 略過置底公告
                if "公告" in title: continue

                post_url = "https://www.ptt.cc" + title_a['href']

                # 進到內文爬取
                post_res = requests.get(post_url, headers=headers, cookies=cookies)
                post_soup = BeautifulSoup(post_res.text, 'html.parser')
                main_content = post_soup.find('div', id='main-content')

                content = ""
                if main_content:
                    # 移除內文中的元數據與推文，只留純文字
                    for meta in main_content.find_all(['div', 'span'], class_=['article-metaline', 'article-metaline-right', 'f2']):
                        meta.decompose()
                    content = main_content.text.strip()

                posts_data.append({
                    "title": title,
                    "content": content,
                    "url": post_url
                })
            except Exception:
                continue

        # 找上一頁的按鈕
        btn_prev = soup.find('a', string='‹ 上頁')
        if btn_prev:
            url = "https://www.ptt.cc" + btn_prev['href']
        else:
            break

    df = pd.DataFrame(posts_data)
    print(f"✅ 爬取完成！共成功補捉到 {len(df)} 篇熱門文章。\n")
    return df

# ==========================================
# 2. TF-IDF 熱詞與情緒分析模組
# ==========================================
def analyze_stock_market_hotspots(df, topk=30):
    if df.empty:
        return "📭 找不到符合條件的文章，無法進行 TF-IDF 分析。"

    corpus = []
    for _, r in df.iterrows():
        corpus.append(f"{r['title']}\n{r['content']}")

    # 過濾掉無意義的股市雜詞
    STOCK_STOPWORDS = ["文章", "網址", "分享", "心得", "看板", "作者", "時間", "推文", "推", "問題", "討論", "心得"]

    def clean_tokenize(text):
        # 抓取 2 到 4 個字的中文字當作詞彙
        words = re.findall(r"[\u4e00-\u9fff]{2,4}", text)
        return [w for w in words if w not in STOCK_STOPWORDS]

    word_counts = Counter()
    doc_frequencies = defaultdict(int)

    for doc in corpus:
        tokens = clean_tokenize(doc)
        word_counts.update(tokens)
        for t in set(tokens):
            doc_frequencies[t] += 1

    total_docs = len(corpus)
    tfidf_scores = {}
    for word, count in word_counts.items():
        df_count = doc_frequencies[word]
        idf = np.log((1 + total_docs) / (1 + df_count)) + 1
        tfidf_scores[word] = count * idf

    sorted_terms = sorted(tfidf_scores.items(), key=lambda x: x[1], reverse=True)[:int(topk)]

    # 簡單的情緒關鍵字標記
    BULL_WORDS = ["大買", "噴噴", "多頭", "利多", "財報好", "買進", "起飛", "爆發", "拉高"]
    BEAR_WORDS = ["慘了", "救命", "崩崩", "綠油油", "停損", "利空", "出貨", "咕嚕", "殺低"]

    report = []
    report.append(f"## 📊 PTT 股版今日關鍵字熱點 (根據 {total_docs} 篇文章分析)")
    report.append("| 名次 | 焦點熱詞 | 討論熱度 (TF-IDF) | 鄉民多空暗示 |")
    report.append("| --- | --- | --- | --- |")

    for rank, (word, score) in enumerate(sorted_terms, 1):
        sentiment = "⚪ 中性/討論"
        if any(b in word for b in BULL_WORDS):
            sentiment = "🔴 偏多 (Bullish)"
        elif any(b in word for b in BEAR_WORDS):
            sentiment = "🟢 偏空 (Bearish)"

        report.append(f"| {rank} | **{word}** | {score:.2f} | {sentiment} |")

    return "\n".join(report)

# ==========================================
# 3. Gemini AI 總結與策略生成模組
# ==========================================
def generate_market_summary(df, tfidf_report):
    if df.empty:
        return "📭 庫存無資料，AI 無法統整。"

    print("🧠 正在將分析報告餵給 Gemini 2.5 Pro，請稍候...")

    # 擷取最後 5 篇文章標題作為範例
    latest_titles = df["title"].tail(5).tolist()
    titles_str = "\n".join([f"- {t}" for t in latest_titles])

    sys_prompt = (
        "你是一位資深的股市分析師與社群觀察家。請根據輸入的 PTT 股版關鍵字分析報告（包含 TF-IDF 分數）"
        "以及最新的文章標題，為交易員撰寫一份簡明扼要的「盤後鄉民風向重點統整」。\n\n"
        "請務必用繁體中文、Markdown 格式回傳，結構如下：\n"
        "### 🔥 今日鄉民最瘋什麼？（核心題材解讀）\n"
        "*(請挑選出 2-3 個今天最火熱的關鍵字或個股，結合當前盤勢解釋為什麼大家在密集討論它)*\n\n"
        "### ⚖️ 市場情緒多空診斷\n"
        "*(鄉民現在的總體氣氛是樂觀追高、恐慌停損，還是處於觀望？請給出你的多空風向評估)*\n\n"
        "### 💡 明日操盤焦點與潛在風險提示\n"
        "*(根據鄉民討論的題材，提醒明天開盤開高或開低時，當沖或波段交易者應該注意什麼風險，例如：小心利多出貨、注意特定均線支撐等)*\n"
    )

    user_content = f"【最新熱詞報告】\n{tfidf_report}\n\n【最新文章標題範例】\n{titles_str}"

    try:
        response = model.generate_content(f"{sys_prompt}\n\n【輸入資料】\n{user_content}")
        return response.text
    except Exception as e:
        return f"⚠️ Gemini 統整失敗，錯誤訊息：{e}"

# ==========================================
# 🚀 4. 主程式一鍵執行區
# ==========================================
if __name__ == "__main__":
    # STEP 1: 抓資料 (你可以自由調整想爬幾頁、或是過濾幾推以上的文章)
    ptt_df = crawl_ptt_stock(pages=3, min_push=10)

    if not ptt_df.empty:
        # STEP 2: 做統計 (產出 TF-IDF 報表)
        tfidf_res = analyze_stock_market_hotspots(ptt_df, topk=20)

        # STEP 3: 丟 AI (讓 Gemini 生成終極大師報告)
        final_ai_report = generate_market_summary(ptt_df, tfidf_res)

        # STEP 4: 在 Colab 裡漂亮地渲染出來
        from IPython.display import display, Markdown
        print("\n" + "="*50 + "\n🎯 【一鍵生成：今日討論版重點報告】 \n" + "="*50 + "\n")
        display(Markdown(final_ai_report))
    else:
        print("❌ 沒抓到資料，請確認網路連線或 PTT 狀態。")

🕷️ 開始爬取 PTT 股版最新 3 頁的文章 (過濾低於 10 推的文章)...
✅ 爬取完成！共成功補捉到 40 篇熱門文章。

🧠 正在將分析報告餵給 Gemini 2.5 Pro，請稍候...


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1491.54ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 784.46ms



🎯 【一鍵生成：今日討論版重點報告】 



身為一位資深的股市分析師與社群觀察家，為您統整 PTT 股版今日的鄉民風向：

### 🔥 今日鄉民最瘋什麼？（核心題材解讀）

今日股版討論的熱點主要圍繞在**個股營收表現**以及**市場潛在的「喊盤」現象**。

*   **月營收與個股表現：** 討論熱度最高的「月營收」、「增減百分」、「增減金額」等關鍵字，與「2059 川湖」、「3653 健策」、「6207 雷科」等具體個股的營收資訊標題高度相關。鄉民們正密集關注這些公司的營收數據，試圖從中尋找股價動能的線索。川湖（2059）5月營收年增率高達172.05%，無疑是今日焦點之一，可預期相關討論將會持續。
*   **「老蘇」與「飆股群組」的潛在警訊：** 雖然「老蘇」（可能指某位分析師或意見領袖）的 TF-IDF 分數來到第五名，且「飆股群組」也進入討論範疇，這暗示著市場上可能存在一些關於特定股票推薦或操作的討論，這需要交易員特別警惕。加上「咕嚕咕嚕」和「完了」的偏空暗示，或許有部分鄉民對市場的某些「喊盤」或過度樂觀的氛圍感到不安。

### ⚖️ 市場情緒多空診斷

**整體而言，市場情緒呈現「觀望為主，部分擔憂」的狀態。**

儘管有多檔個股的營收數據不錯，吸引了關注，但「媽媽」、「了吧」、「丸子」、「公公」、「穩了」、「開始」、「今天」、「外資」等許多高熱度詞彙都顯示為「中性/討論」，缺乏強烈的買進或賣出訊號。

值得注意的是，**「咕嚕咕嚕」的偏空暗示，以及「完了」的出現，顯示有一部分鄉民對後續盤勢抱持謹慎甚至偏空的態度。** 這可能與部分個股的獲利了結、對「飆股群組」的疑慮，或是對大盤指數缺乏明確方向的擔憂有關。

### 💡 明日操盤焦點與潛在風險提示

根據今日鄉民的討論風向，明日操盤應留意以下幾點：

*   **開盤時段：**
    *   **留意利多出貨：** 若今日營收表現亮眼的個股（如川湖）開盤直接跳空上漲，交易員應特別警惕「利多出貨」的可能性。部分鄉民可能已經在營收公佈前佈局，開盤後可能面臨獲利了結賣壓。
    *   **關注「老蘇」與「飆股群組」的後續影響：** 若明日有相關討論延燒，需留意是否有特定股票被過度拉抬的風險。
*   **均線支撐與壓力：**
    *   **多頭部位：** 若看好營收題材，應關注個股昨日收盤價或關鍵均線（如 5 日、10 日均線）的支撐。
    *   **空方部位：** 若市場情緒轉為偏空，需留意大盤或個股可能面臨的壓力點，特別是近期漲多且獲利了結賣壓湧現的股票。
*   **風險提示：**
    *   **「咕嚕咕嚕」與「完了」的警訊：** 這兩個偏空關鍵字提醒我們，市場上存在潛在的風險，不宜過度追高。若盤勢出現明顯轉弱跡象，應及時出場。
    *   **整體討論熱度分散：** 除了營收題材外，其他討論多為中性，顯示市場缺乏明顯的「主流」題材帶動，操作上應保持謹慎，避免盲目跟單。

## 常見錯誤檢查

如果 PTT 資料沒有寫回 Google Sheet，請依序檢查：

1. 是否有成功印出 `已開啟試算表`，且名稱正確。
2. `SHEET_URL` 是否是你要寫入的那一份 Google Sheet。
3. Google Sheet 權限是否允許目前 Colab 登入的 Google 帳號編輯。
4. 是否執行到「執行爬蟲並寫入 Google Sheet」那一格。
5. 是否有看到 `寫入驗證成功`。
6. RAG 要從 `rag_source_df = read_sheet_df(...)` 開始，確保資料來源是 Google Sheet，而不是記憶體中的暫存變數。
